In [ ]:
import h5py as h5
import json
import os
from glob import glob
import sys
import tifffile
import pandas as pd
import numpy as np
import torch

sys.path.append('/home/stumberger/fish-pipelines/')
from fish_utils.resave import resave_h5, remove_tifs
from fish_utils.spot_detection import read_parameters, make_fiji_command, detect_spots, combine_csv, create_folder, plot_detections
from fish_utils.spot_analysis import add_cell_info, get_sensitivity

from natsort import natsorted
import matplotlib.pyplot as plt
from skimage.io import imread

from cellpose import models, io
from cellpose.io import imread
from cellpose import plot

# Resave h5 as tif

In [ ]:
# folder containing the images
files = "/data/agl_data/microfluidics/20230830_hyb+strip_seq1-seq3/"

resave_h5(files) 

# make sure to delete the tifs after you are done with the analysis!!!

# Detect spots 
Before running this, open one of your images in fiji, run RS-FISH on it, and save the Log file with the spot detection parameters - IMPORTAINT!

In [ ]:
path = "/data/agl_data/microfluidics/20230830_hyb+strip_seq1-seq3/"

detection_settings = ["/data/agl_data/microfluidics/20230818_hyb_stripping_STAR635P_seq3/detection_settings.txt"]
channels = [0] # which channel to detect spots in (counting starts at 0!)

# run
detect_spots(path,detection_settings,channels,
            macro_path = "/home/stumberger/fish-pipelines/fish_utils/RS_macro_param.ijm",
            fiji_path = "/home/stumberger/tools/Fiji.app/ImageJ-linux64")

# combine all csvs in a folder into 1
combine_csv(path)

# Plot the detections

In [ ]:
# check how well the spot detection worked
path = "/data/agl_data/microfluidics/20230830_hyb+strip_seq1-seq3/"
path_spots= None #if None defaults to /.../detections/merge.csv 
out_folder = None #if None defaults to /.../detections/vis
channels = [0] # which channel to plot spots for (counting starts at 0!)

plot_detections(path,channels)

# Segmentation

In [ ]:
# model = models.Cellpose(model_type='cyto', gpu=True)
model = models.CellposeModel(model_type = "microfluidic_20230828") # device = cudagpunamehere

files = glob("/data/agl_data/microfluidics/20230830_hyb+strip_seq1-seq3/tif/*_ch0.tif")
out_folder = "/data/agl_data/microfluidics/20230830_hyb+strip_seq1-seq3/segmentation/vis/"

chan = [[0,0]]
diams = 115

# or in a loop
for filename in files:
    
    out = filename.replace("/tif", "/segmentation")
    create_folder(out_folder)
    img = io.imread(filename).max(axis=0)
    
    masks, flows, styles= model.eval(img, channels=chan,flow_threshold=0.5)

    # save results so you can load in gui
    io.masks_flows_to_seg(img, masks, flows, diams, out)

    # save results as png
    io.save_to_png(img, masks, flows, out)
    
    # plot segmentation o check
    fig = plt.figure(figsize=(12,3.5))
    plot.show_segmentation(fig, img, masks, flows[0], channels=chan)
    plt.tight_layout()
    fig.savefig(f"{out_folder}/{os.path.basename(out)}.png",dpi=300)
    plt.close(fig)

# Remove spots not in segmentation

In [ ]:
# add cell information to individual spot
# remove spots which are not in segmentation
path = "/data/agl_data/microfluidics/20230830_hyb+strip_seq1-seq3"
out_file = f"{path}/spot_stats.csv"
out_file1 = f"{path}/sensitivity.csv"

add_cell_info(path,out_file, mask_ending="_cp_masks", filter=True)
# get_sensitivity(out_file,out_file1)

# can now plot the spots_detection again if desired

# Remove tif folder

In [ ]:
# please always remove tifs afetr you are done with the analysis to save space
folder = "/data/agl_data/microfluidics/20230818_hyb_stripping_STAR635P_seq3/tir/"

remove_tifs(folder)

# Make intensity graphs for each run

In [ ]:
import re
import seaborn as sns
spots = pd.read_csv("/data/agl_data/microfluidics/20230830_hyb+strip_seq1-seq3/detections/merge.csv")
spots['run'] = spots['img'].apply(lambda x: re.search(r"run(\d+)", x).group(0) if re.search(r"run(\d+)_", x) else None)
spots.groupby('run')

# sort the x labels
sorted_labs = sorted(spots['run'].dropna().unique())

# create boxplot
plt.figure(figsize=(10, 6))
ax=sns.boxplot(x='run', y='intensity', data=spots, color = "lightblue",order=sorted_labs)
plt.xlabel('timepoint')
plt.ylabel('intensity')
plt.title('Intensity of spots in each run')

# Calculate number of obs per group & median to position labels
medians = spots.groupby(['run'])['intensity'].median().values
nobs = spots['run'].value_counts().sort_index().values
nobs = [str(x) for x in nobs.tolist()]
nobs = ["n: " + i for i in nobs]
nobs

# Add it to the plot
pos = range(len(nobs))
for tick,label in zip(pos,ax.get_xticklabels()):
    ax.text(pos[tick],
            32800 - 8,
            nobs[tick],
            horizontalalignment='center',
            size='medium',
            color='black')
 
plt.show()